# InverseRG: 2D Compact U(1) Naive Pipeline

This notebook presents the full naive MCRG pipeline for 2D compact U(1) lattice gauge theory:

1. **HMC Sampling** -- generate fine-lattice configurations with Hybrid Monte Carlo
2. **Naive 2x2 Blocking** -- produce coarse configurations by summing consecutive link phases
3. **Independent Coarse HMC** -- generate coarse configurations directly from the Wilson action at $\beta_c = \beta_f / 4$
4. **Distribution Comparison** -- compare blocked-fine vs coarse-HMC ensembles at the distribution level

**Parameters:**
- Fine lattice: $L = 32$, $\beta_f = 4.0$, 1000 configurations
- Coarse lattice: $L = 16$, $\beta_c = 1.0$, 1000 configurations

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from inverserg.actions import LocalWilsonLoopAction
from inverserg.baselines import tree_level_coarse_beta
from inverserg.blocking import NaiveBlocker
from inverserg.hmc import HMCU1Sampler, HMCDiagnostics
from inverserg.lattice import plaquette_angles, regularize
from inverserg.measurements import (
    measurement_samples,
    plaquette_theory,
    topological_susceptibility_theory,
    autocorrelation_from_topo,
)
from inverserg.diagnostics import analyze_distribution_consistency

torch.manual_seed(42)

FINE_L = 32
FINE_BETA = 4.0
COARSE_L = FINE_L // 2
COARSE_BETA = tree_level_coarse_beta(FINE_BETA)
N_CONFIGS = 1000
HMC_STEPS = 10
HMC_STEP_SIZE = 0.1
BURN_IN = 200
THIN = 4

print(f"Fine: L={FINE_L}, beta={FINE_BETA}")
print(f"Coarse: L={COARSE_L}, beta={COARSE_BETA}")
print(f"Configs per ensemble: {N_CONFIGS}")

---
## Section A: HMC Sampling for 2D Compact U(1)

### The Fine-Lattice Theory

We simulate 2D compact U(1) lattice gauge theory on a periodic $L \times L$ square lattice. The degrees of freedom are link angles $\theta_\mu(x) \in [-\pi, \pi)$ for each link, where $\mu \in \{0, 1\}$ labels the direction (x or y).

The **Wilson plaquette action** is:
$$S[\theta] = -\beta \sum_{\text{plaquettes}} \cos(\theta_P)$$
where the plaquette angle $\theta_P$ is the oriented sum of link angles around each elementary square.

### HMC with Omelyan Integrator

We sample configurations using Hybrid Monte Carlo (HMC) with a second-order Omelyan integrator ($\lambda = 0.1932$). Each HMC trajectory:
1. Draws conjugate momenta $\pi$ from a Gaussian
2. Evolves $(\theta, \pi)$ via the Omelyan integrator for `n_steps` steps
3. Applies Metropolis accept/reject on the Hamiltonian $H = S(\theta) + \frac{1}{2}\pi^2$

### Exact Results

For 2D compact U(1) the mean plaquette has an exact analytic value:
$$\langle \cos(\theta_P) \rangle = \frac{I_1(\beta)}{I_0(\beta)}$$
where $I_n$ are modified Bessel functions.

In [ ]:
fine_action = LocalWilsonLoopAction.wilson(FINE_BETA)
fine_sampler = HMCU1Sampler(
    lattice_size=FINE_L,
    action=fine_action,
    n_steps=HMC_STEPS,
    step_size=HMC_STEP_SIZE,
)

print(f"Generating {N_CONFIGS} fine configs on {FINE_L}x{FINE_L} lattice at beta={FINE_BETA}...")
print(f"HMC: n_steps={HMC_STEPS}, step_size={HMC_STEP_SIZE}, burn_in={BURN_IN}, thin={THIN}")
print(f"Total Metropolis steps: {BURN_IN + N_CONFIGS * THIN}")

fine_configs, fine_acc, _, fine_diag = fine_sampler.sample_with_diagnostics(
    n_samples=N_CONFIGS,
    burn_in=BURN_IN,
    thin=THIN,
)
print(f"Done. Shape: {fine_configs.shape}")
print(f"Acceptance rate: {fine_acc:.4f}")

### Fine HMC Diagnostics

The 4-panel figure below shows the key diagnostics for the fine-lattice HMC chain:
- **Plaquette vs iteration**: thermalization (blue) followed by sampling (orange), with the exact theoretical value (red dashed)
- **Action vs iteration**: the Wilson action value $S(\theta)$ with its mean (red dashed)
- **Topological charge vs iteration**: $Q = \frac{1}{2\pi}\sum_P \theta_P$ (rounded to nearest integer)
- **Autocorrelation**: topological charge autocorrelation estimated via the susceptibility method

In [ ]:
def plot_hmc_diagnostics(diag, beta, L, title_prefix=""):
    """4-panel HMC diagnostics plot."""
    volume = L * L
    burn = diag.burn_in_length
    n_total = len(diag.plaquette_history)
    iters = np.arange(n_total)
    plaq = np.array(diag.plaquette_history)
    action = np.array(diag.hamiltonian_history)
    topo = np.array(diag.topological_charge_history)

    plaq_exact = plaquette_theory(beta)
    chi_exact = topological_susceptibility_theory(beta)

    max_lag = min(200, n_total // 4)
    autocor = autocorrelation_from_topo(topo[burn:], max_lag, beta, volume)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fs = 14

    ax = axes[0, 0]
    ax.plot(iters[:burn], plaq[:burn], color='steelblue', alpha=0.7, label='Thermalization')
    ax.plot(iters[burn:], plaq[burn:], color='darkorange', alpha=0.7, label='Sampling')
    ax.axhline(plaq_exact, color='red', linestyle='--', linewidth=1.5,
               label=f'Theory $I_1/I_0 = {plaq_exact:.6f}$')
    ax.axvline(burn, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Iteration', fontsize=fs)
    ax.set_ylabel('Mean Plaquette', fontsize=fs)
    ax.set_title(f'{title_prefix}Plaquette vs Iteration', fontsize=fs)
    ax.legend(fontsize=fs - 2)
    ax.tick_params(labelsize=fs - 2)
    ax.grid(linestyle=':', alpha=0.5)

    ax = axes[0, 1]
    ax.plot(iters, action, color='steelblue', alpha=0.5, linewidth=0.5)
    ax.axhline(np.mean(action[burn:]), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean = {np.mean(action[burn:]):.1f}')
    ax.axvline(burn, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Iteration', fontsize=fs)
    ax.set_ylabel(r'Action $S(\theta)$', fontsize=fs)
    ax.set_title(f'{title_prefix}Action vs Iteration', fontsize=fs)
    ax.legend(fontsize=fs - 2)
    ax.tick_params(labelsize=fs - 2)
    ax.grid(linestyle=':', alpha=0.5)

    ax = axes[1, 0]
    ax.plot(iters[burn:], topo[burn:], color='steelblue', alpha=0.7,
            marker='o', markersize=1, linewidth=0.5)
    ax.axhline(np.mean(topo[burn:]), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean = {np.mean(topo[burn:]):.3f}')
    ax.set_xlabel('Iteration', fontsize=fs)
    ax.set_ylabel(r'Topological Charge $Q$', fontsize=fs)
    ax.set_title(f'{title_prefix}Topological Charge vs Iteration', fontsize=fs)
    ax.legend(fontsize=fs - 2)
    ax.tick_params(labelsize=fs - 2)
    ax.grid(linestyle=':', alpha=0.5)

    ax = axes[1, 1]
    ax.plot(range(len(autocor)), autocor, color='steelblue', marker='o', markersize=2)
    ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
    ax.set_xlabel('Lag (MDTU)', fontsize=fs)
    ax.set_ylabel('Autocorrelation', fontsize=fs)
    ax.set_title(f'{title_prefix}Topological Charge Autocorrelation', fontsize=fs)
    ax.tick_params(labelsize=fs - 2)
    ax.grid(linestyle=':', alpha=0.5)

    fig.suptitle(f'{title_prefix}L={L}, $\\beta$={beta}', fontsize=fs + 2, fontweight='bold', y=1.01)
    fig.tight_layout()
    plt.show()

    plaq_measured = np.mean(plaq[burn:])
    topo_arr = np.array(topo[burn:])
    suscept_measured = np.mean(topo_arr**2) / volume
    accept_total = np.mean(diag.acceptance_history)

    print(f"--- {title_prefix}Summary (L={L}, beta={beta}) ---")
    print(f"Acceptance rate: {accept_total:.4f}")
    print(f"Mean plaquette (measured): {plaq_measured:.6f}")
    print(f"Mean plaquette (theory):   {plaq_exact:.6f}")
    print(f"Difference:                {plaq_measured - plaq_exact:.6f}")
    print(f"Topological susceptibility (measured): {suscept_measured:.6f}")
    print(f"Topological susceptibility (theory):   {chi_exact:.6f}")


plot_hmc_diagnostics(fine_diag, FINE_BETA, FINE_L, title_prefix="Fine ")

---
## Section B: Naive MCRG Blocking

### The 2x2 Blocking Map

We block the $32 \times 32$ fine lattice down to a $16 \times 16$ coarse lattice using the **naive 2x2 blocking** scheme:

For a 2x2 block starting at fine site $(2i, 2j)$:
- **Coarse x-link**: $\Theta_x(i,j) = \mathrm{regularize}\big(\theta_x(2i, 2j) + \theta_x(2i{+}1, 2j)\big)$
- **Coarse y-link**: $\Theta_y(i,j) = \mathrm{regularize}\big(\theta_y(2i, 2j) + \theta_y(2i, 2j{+}1)\big)$

This sums two consecutive fine-link phases along the same direction, then wraps the result back to $[-\pi, \pi]$.

### Why This Preserves Gauge Symmetry

In terms of link variables $U_\mu(x) = e^{i\theta_\mu(x)}$, summing the phases is equivalent to multiplying the link variables:
$$e^{i\Theta_x(i,j)} = e^{i\theta_x(2i,2j)} \cdot e^{i\theta_x(2i+1,2j)}$$

Under a gauge transformation $\theta_\mu(x) \to \theta_\mu(x) + \alpha(x) - \alpha(x+\hat\mu)$, the coarse link transforms as:
$$\Theta_x(i,j) \to \Theta_x(i,j) + \alpha(2i, 2j) - \alpha(2i+2, 2j)$$

which is a gauge transformation on the coarse lattice with $A(i,j) = \alpha(2i, 2j)$.

In [ ]:
blocker = NaiveBlocker()
blocked_configs = blocker(fine_configs)
print(f"Fine configs shape:    {fine_configs.shape}")
print(f"Blocked configs shape: {blocked_configs.shape}")

### Visualization: Fine vs Blocked Plaquette Fields

Below we show the plaquette angle field $\theta_P(x)$ for a single configuration before and after blocking. The plaquette angle is the oriented sum of link angles around each elementary square, wrapped to $[-\pi, \pi]$.

In [ ]:
idx = 0
fine_plaq = plaquette_angles(fine_configs[idx]).detach().numpy()
blocked_plaq = plaquette_angles(blocked_configs[idx]).detach().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

im1 = ax1.imshow(fine_plaq, cmap='twilight', vmin=-np.pi, vmax=np.pi, origin='lower')
ax1.set_title(f'Fine Plaquette Field ({FINE_L}x{FINE_L}, $\beta$={FINE_BETA})', fontsize=13)
ax1.set_xlabel('y', fontsize=12)
ax1.set_ylabel('x', fontsize=12)
plt.colorbar(im1, ax=ax1, label=r'$\theta_P$')

im2 = ax2.imshow(blocked_plaq, cmap='twilight', vmin=-np.pi, vmax=np.pi, origin='lower')
ax2.set_title(f'Blocked Plaquette Field ({COARSE_L}x{COARSE_L})', fontsize=13)
ax2.set_xlabel('y', fontsize=12)
ax2.set_ylabel('x', fontsize=12)
plt.colorbar(im2, ax=ax2, label=r'$\theta_P$')

fig.tight_layout()
plt.show()

### Plaquette Angle Distributions: Fine vs Blocked

The histogram below shows how the distribution of plaquette angles shifts after blocking. At large $\beta$ (ordered phase), fine plaquettes are concentrated near zero; blocking broadens the distribution as we move to a coarser lattice at effectively lower $\beta$.

In [ ]:
all_fine_plaq = plaquette_angles(fine_configs).detach().numpy().flatten()
all_blocked_plaq = plaquette_angles(blocked_configs).detach().numpy().flatten()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_fine_plaq, bins=80, density=True, alpha=0.6, color='steelblue',
        label=f'Fine ({FINE_L}x{FINE_L}, $\beta$={FINE_BETA})')
ax.hist(all_blocked_plaq, bins=80, density=True, alpha=0.6, color='darkorange',
        label=f'Blocked ({COARSE_L}x{COARSE_L})')
ax.set_xlabel(r'Plaquette angle $\theta_P$', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title('Plaquette Angle Distribution: Fine vs Blocked', fontsize=14)
ax.legend(fontsize=12)
ax.tick_params(labelsize=11)
ax.grid(linestyle=':', alpha=0.5)
fig.tight_layout()
plt.show()

print(f"Fine: mean cos(theta_P) = {np.mean(np.cos(all_fine_plaq)):.6f}  (theory: {plaquette_theory(FINE_BETA):.6f})")
print(f"Blocked: mean cos(theta_P) = {np.mean(np.cos(all_blocked_plaq)):.6f}  (theory at beta_c={COARSE_BETA}: {plaquette_theory(COARSE_BETA):.6f})")

---
## Section C: Independent Coarse HMC

We now generate an independent ensemble on the coarse lattice ($L=16$) using the Wilson action at $\beta_c = \beta_f / 4 = 1.0$.

**If the naive blocking + tree-level $\beta_c$ hypothesis is correct**, the distributions of observables measured on the blocked-fine ensemble and the coarse-HMC ensemble should agree.

In [ ]:
coarse_action = LocalWilsonLoopAction.wilson(COARSE_BETA)
coarse_sampler = HMCU1Sampler(
    lattice_size=COARSE_L,
    action=coarse_action,
    n_steps=HMC_STEPS,
    step_size=HMC_STEP_SIZE,
)

print(f"Generating {N_CONFIGS} coarse configs on {COARSE_L}x{COARSE_L} lattice at beta={COARSE_BETA}...")

coarse_configs, coarse_acc, _, coarse_diag = coarse_sampler.sample_with_diagnostics(
    n_samples=N_CONFIGS,
    burn_in=BURN_IN,
    thin=THIN,
)
print(f"Done. Shape: {coarse_configs.shape}")
print(f"Acceptance rate: {coarse_acc:.4f}")

In [ ]:
plot_hmc_diagnostics(coarse_diag, COARSE_BETA, COARSE_L, title_prefix="Coarse ")

---
## Section D: Blocked-Fine vs Coarse-HMC Comparison

This is the key section. We compare the distributions of physical observables measured on:
1. **Blocked-fine ensemble** -- fine configs blocked to the coarse lattice via naive 2x2 blocking
2. **Coarse-HMC ensemble** -- independently generated coarse configs from the Wilson action at $\beta_c = 1.0$

For each observable, we show:
- Overlay histograms
- Empirical CDF comparison
- Two-sample Kolmogorov-Smirnov (KS) test

### Observables
| Observable | Description |
|---|---|
| `plaquette` | $\langle \cos(\theta_P) \rangle$ per configuration |
| `rectangle_x` | $\langle \cos(\theta_{2 \times 1}) \rangle$ per configuration |
| `rectangle_y` | $\langle \cos(\theta_{1 \times 2}) \rangle$ per configuration |
| `wilson_2x2` | $\langle \cos(\theta_{2 \times 2}) \rangle$ per configuration |
| `topological_charge` | $Q = \mathrm{round}\big(\frac{1}{2\pi}\sum_P \theta_P\big)$ per configuration |

In [ ]:
MEASUREMENT_NAMES = ("plaquette", "rectangle_x", "rectangle_y", "wilson_2x2", "topological_charge")

diagnostics_list, merged_samples = analyze_distribution_consistency(
    blocked_configs,
    coarse_configs,
    measurement_names=MEASUREMENT_NAMES,
)

n_meas = len(MEASUREMENT_NAMES)
fig, axes = plt.subplots(n_meas, 2, figsize=(14, 4 * n_meas))

for row, (diag_item, mname) in enumerate(zip(diagnostics_list, MEASUREMENT_NAMES)):
    blocked_vals = merged_samples[f"blocked_{mname}"].numpy()
    coarse_vals = merged_samples[f"coarse_{mname}"].numpy()

    ax_hist = axes[row, 0]
    ax_hist.hist(blocked_vals, bins=30, density=True, alpha=0.6, color='steelblue', label='Blocked-fine')
    ax_hist.hist(coarse_vals, bins=30, density=True, alpha=0.6, color='darkorange', label='Coarse HMC')
    verdict = 'PASS' if diag_item.consistent else 'FAIL'
    ax_hist.set_title(f'{mname} -- KS={diag_item.ks_statistic:.4f} (crit={diag_item.ks_critical_value:.4f}, {verdict})',
                      fontsize=12)
    ax_hist.legend(fontsize=10)
    ax_hist.set_ylabel('Density', fontsize=11)
    ax_hist.grid(linestyle=':', alpha=0.4)

    ax_cdf = axes[row, 1]
    blocked_sorted = np.sort(blocked_vals)
    coarse_sorted = np.sort(coarse_vals)
    blocked_cdf = np.arange(1, len(blocked_sorted) + 1) / len(blocked_sorted)
    coarse_cdf = np.arange(1, len(coarse_sorted) + 1) / len(coarse_sorted)
    ax_cdf.plot(blocked_sorted, blocked_cdf, color='steelblue', linewidth=1.5, label='Blocked-fine')
    ax_cdf.plot(coarse_sorted, coarse_cdf, color='darkorange', linewidth=1.5, label='Coarse HMC')
    ax_cdf.set_title(f'{mname} -- Empirical CDF', fontsize=12)
    ax_cdf.legend(fontsize=10)
    ax_cdf.set_ylabel('CDF', fontsize=11)
    ax_cdf.grid(linestyle=':', alpha=0.4)

fig.tight_layout()
plt.show()

### Summary Table

In [ ]:
header = "| Measurement | Blocked Mean | Coarse Mean | Blocked Std | Coarse Std | KS Stat | KS Critical | Verdict |"
sep    = "|---|---:|---:|---:|---:|---:|---:|---|"
rows = [header, sep]

for d in diagnostics_list:
    verdict = "PASS" if d.consistent else "**FAIL**"
    rows.append(
        f"| {d.measurement} | {d.blocked_mean:.6f} | {d.coarse_mean:.6f} | "
        f"{d.blocked_std:.6f} | {d.coarse_std:.6f} | {d.ks_statistic:.4f} | "
        f"{d.ks_critical_value:.4f} | {verdict} |"
    )

display(Markdown("\n".join(rows)))

---
## Section E: Summary and Next Steps

### What We Did

1. Generated 1000 fine-lattice configurations ($32 \times 32$, $\beta_f = 4.0$) using HMC with Omelyan integrator.
2. Applied naive 2x2 blocking (sum two consecutive link phases, regularize) to obtain 1000 blocked-coarse configurations on $16 \times 16$.
3. Generated 1000 independent coarse configurations ($16 \times 16$, $\beta_c = 1.0$) via HMC.
4. Compared the two coarse ensembles at the distribution level using histograms, CDFs, and KS tests for five observables.

### Interpretation

The naive blocking with tree-level $\beta_c = \beta_f / 4$ provides a first approximation. Discrepancies between the blocked-fine and coarse-HMC distributions reveal where the naive approach breaks down -- this motivates the next phase.

### Next Steps

1. **Learned Blocking**: Replace the naive blocker with a learnable gauge-covariant path-weight blocker that can optimize path combinations.
2. **Generalized Coarse Action**: Extend the coarse action beyond pure Wilson by adding rectangle and larger loop terms.
3. **Joint Training**: Simultaneously optimize the blocking map and coarse action coefficients using distribution-matching losses (MMD + contrastive).
4. **Scaling**: Test at larger lattice sizes and multiple $\beta_f$ values to validate universality of the learned map.